<a href="https://colab.research.google.com/github/BarbaraEstimable/IA2_projet1/blob/Impl%C3%A9mentation-graphique-et-comparaison-des-algorithmes/projet1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projet 1 - Plus Court Chemin entre Villes




###Contexte

---



On souhaite modéliser un réseau routier entre plusieurs villes nord-américaines
sous la forme d’un graphe orienté pondéré. Chaque ville est un noeud, chaque
route entre deux villes est un arc orienté dont le poids représente la distance en
kilomètres.
L’objectif est de trouver le plus court chemin entre une ville de départ et
une ville d’arrivée, en utilisant des algorithmes de plus court chemin, puis de
comparer leurs performances.

###Objectif

---



À partir du jeu de données fourni ci-dessous (villes, coordonnées, routes et distances),
implémenter une structure de graphe orienté pondéré, puis utiliser
au moins deux algorithmes parmi Dijkstra, A* et Bellman-Ford pour trouver
le plus court chemin entre deux villes.
Pour chacun des algorithmes, afficher dans la console les performances (nombre
de noeuds explorés, coût total du chemin, temps d’exécution, …).

###Directives

---





*   Nous travaillerons avec un graphe orienté pondéré (le poids d’un arc
représente une distance en km entre deux villes)
*   Une Ville est un objet possédant un nom et des coordonnées géographiques
(x, y) utilisées pour le calcul de l’heuristique de A*


*   Un Arc est orienté : si une route Montréal → Toronto existe avec un
poids donné, cela ne signifie pas que Toronto → Montréal existe avec le
même poids
*   Le jeu de données est entièrement fourni : vous n’avez pas à inventer
les villes, coordonnées ou distances


*   Une implémentation graphique devra être réalisée : visualisation du
graphe avec les arcs fléchés, les poids, et le chemin solution mis en évidence.
(Utilisation de la librairie de votre choix, Graphviz ou Matplotlib ou autre)
*   Implémenter au moins 2 algorithmes parmi : Dijkstra, A*, Bellman-
Ford


*   Heuristique de A* : Distance Euclidienne entre les coordonnées (x, y)
des villes
*   Choisir au moins 3 métriques pertinentes : nombre de noeuds explorés,
coût total du chemin (km), temps d’exécution, …


*   Une analyse des résultats est attendue : discuter des différences de
performance entre les algorithmes, et expliquer dans quels cas chacun est
préférable











 **Pour l’analyse des résultats prenez le chemin entre
Québec et Buffalo**

### Données fournies

In [104]:
VILLES = {
  "Montreal": (45.30, -73.35),
  "Quebec": (46.81, -71.21),
  "Ottawa": (45.42, -75.70),
  "Toronto": (43.65, -79.38),
  "Buffalo": (42.89, -78.87),
  "Boston": (42.36, -71.06),
  "New York": (40.71, -74.01),
  "Chicago": (41.88, -87.63),
}

ROUTES = [
  # Depuis Montreal
  ("Montreal", "Quebec", 250),
  ("Montreal", "Ottawa", 200),
  ("Montreal", "Boston", 435),
  ("Montreal", "New York", 595),

  # Depuis Quebec
  ("Quebec", "Montreal", 255),
  ("Quebec", "Boston", 650),

  # Depuis Ottawa
  ("Ottawa", "Montreal", 195),
  ("Ottawa", "Toronto", 450),

  # Depuis Toronto
  ("Toronto", "Ottawa", 445),
  ("Toronto", "Buffalo", 155),
  ("Toronto", "Chicago", 840),

  # Depuis Buffalo
  ("Buffalo", "Toronto", 160),
  ("Buffalo", "New York", 590),
  ("Buffalo", "Boston", 700),
  ("Buffalo", "Chicago", 860),

  # Depuis Boston
  ("Boston", "New York", 345),
  ("Boston", "Montreal", 440),
  ("Boston", "Buffalo", 695),

  # Depuis New York
  ("New York", "Boston", 350),
  ("New York", "Buffalo", 585),
  ("New York", "Chicago", 1270),

  # Depuis Chicago
  ("Chicago", "Toronto", 835),
  ("Chicago", "Buffalo", 855),
  ("Chicago", "New York", 1275),
]

### Importations

In [105]:
import heapq
import time
import graphviz
import math

### Les class

In [106]:
class Ville:
    def __init__(self, nom, x, y):
        self.nom = nom
        self.x = x  # longitude approximative en degrés
        self.y = y  # latitude approximative en degrés


class Graphe:
    def __init__(self, liste_villes):
        graphe_liste_adjacence = {}
        for ville in liste_villes:
            graphe_liste_adjacence.update({ville : []})

        self.graphe_liste_adjacence = graphe_liste_adjacence
        self.villes = {}

    def ajoute_ville(self, nom, x, y):
        self.villes[nom] = Ville(nom, x, y)

    def ajoute_arc(self, noeud1, noeud2, poids):
        self.graphe_liste_adjacence[noeud1].append((noeud2, poids))


    def sont_voisins(self, noeud1, noeud2) -> bool:
        return any(v==noeud2 for v, _ in self.graphe_liste_adjacence[noeud1])

    def voisins(self, noeud) -> list:
        return self.graphe_liste_adjacence[noeud]


In [107]:
# La construction du graphe
def construction_graphe():
    liste_villes = list(VILLES.keys())
    graphe = Graphe(liste_villes)
    for nom_ville, coordonnees in VILLES.items():
        graphe.ajoute_ville(nom_ville, coordonnees[1], coordonnees[0])

    for origine, destination, distance in ROUTES:
        graphe.ajoute_arc(origine, destination, distance)
    return graphe

In [108]:
# Le chemin
def remonte_chemin(depart, arrivee, parent):
    sommet = arrivee
    chemin = [arrivee]
    while sommet != depart:
        sommet = parent[sommet]
        chemin.append(sommet)
    chemin.reverse()
    return chemin


### Algorithme Dijkstra

In [109]:
def dijkstra(grille, depart, arrivee):

    # Le temps de départ
    debut = time.perf_counter()

    # Initialisation des distances
    distances     = {}
    for sommet in grille.graphe_liste_adjacence:
        distances[sommet] = float('inf')

    predecesseurs = {depart: None}
    traites       = []

    # Les noeuds explorés
    explores      = 0

    distances[depart] = 0
    file = [(0, depart)]




    while file:
        distance_actuelle, sommet_actuel = heapq.heappop(file)

        if sommet_actuel in traites:
          continue

        traites.append(sommet_actuel)
        explores += 1

        if sommet_actuel == arrivee:
            break

        voisins = grille.voisins(sommet_actuel)

        for voisin, poids in voisins:
            nouveau_cout = distance_actuelle + poids
            if nouveau_cout < distances[voisin]:
                distances[voisin]     = nouveau_cout
                predecesseurs[voisin] = sommet_actuel
                heapq.heappush(file, (nouveau_cout, voisin))
    # Le temps fin
    fin = time.perf_counter()
    chemin = remonte_chemin(depart, arrivee, predecesseurs)

    return {
        'chemin': chemin,
        'distance': distances[arrivee],
        'temps': (fin - debut) * 1000,
        'explores': explores
    }

### L'algorithme A*

In [110]:
# Heuristique de A*

def heuristique(ville_depart, ville_arrivee, grille):
      ville1 = grille.villes[ville_depart]
      ville2 = grille.villes[ville_arrivee]
      return math.sqrt((ville2.x - ville1.x)**2 + (ville2.y - ville1.y)**2)



def a_star(grille, depart, arrivee):
    # Le temps
    debut = time.perf_counter()

    g_score = {}
    for sommet in grille.graphe_liste_adjacence:
        g_score[sommet] = float('inf')
    g_score[depart]  = 0
    predecesseurs = {depart: None}
    explores    = 0
    traites = []

    f_score = 0 + heuristique(depart, arrivee, grille)
    # (f, g, noeud)
    file = [(f_score, depart)]

    while file:
        f_actuel, sommet_actuel = heapq.heappop(file)

        if sommet_actuel in traites:
            continue

        traites.append(sommet_actuel)
        explores += 1

        if sommet_actuel == arrivee:
            break

        voisins = grille.voisins(sommet_actuel)

        for voisin, poids in voisins:
            nouveau_g_score = g_score[sommet_actuel] + poids
            if nouveau_g_score < g_score[voisin]:
                g_score[voisin] = nouveau_g_score
                predecesseurs[voisin] = sommet_actuel
                f_voisin = nouveau_g_score + heuristique(voisin, arrivee, grille)
                heapq.heappush(file, (f_voisin, voisin))


    fin = time.perf_counter()
    chemin = remonte_chemin(depart, arrivee, predecesseurs)

    return {
        'chemin': chemin,
        'distance': g_score[arrivee],
        'temps': (fin - debut)*1000,
        'explores': explores
    }

### Implémentation graphique avec Graphviz

In [111]:
def visualiser_graphe(chemin_optimal):
    dot = graphviz.Graph('Reseau Routier Nord-Américains')
    dot.attr(rankdir='LR', bgcolor='#1a1a2e')
    dot.attr('node',
        shape='circle',
        style='filled',
        fontname='Arial-Bold',
        fontsize='11',
        fontcolor='white',
        fillcolor='#2980b9',
        width='1.0'
    )
    dot.attr('edge',
        fontname='Arial',
        fontsize='11',
        fontcolor='#a0aec0',
        color='#4a90d9',
        penwidth='1.8'
    )

    for ville in VILLES.keys():
        dot.node(ville, label=ville)

    added_edges = set()
    if chemin_optimal:
        for i in range(len(chemin_optimal) - 1):
            added_edges.add((chemin_optimal[i], chemin_optimal[i + 1]))

    for origine, destination, distance in ROUTES:
        label_distance = f"{distance} km"
        if (origine, destination) in added_edges:
            dot.edge(origine, destination, label=label_distance, color='#2ecc71')
        else:
            dot.edge(origine, destination, label=label_distance, color='#e74c3c')

    dot.render('resultat_reseau_routier ', format='pdf', cleanup=True, view=True)

    print("Graphe généré : resultat_reseau_routier.pdf\n")

### Comparaison des algorithmes Dijkstra et A_star

In [112]:
def comparer():
    print("=" * 50)
    print("   COMPARAISON A* vs DIJKSTRA")
    print("=" * 50)

    # A*
    resultat_astar = a_star(grille, depart, arrivee)
    temps_astar  = resultat_astar['temps']
    chemin_astar = resultat_astar['chemin']
    dist_astar   = resultat_astar['distance']
    exp_astar    = resultat_astar['explores']

    # Dijkstra

    resultat_dijk = dijkstra(grille, depart, arrivee)
    temps_dijk = resultat_dijk['temps']
    chemin_dijk = resultat_dijk['chemin']
    dist_dijk   = resultat_dijk['distance']
    exp_dijk    = resultat_dijk['explores']

    # Tableau comparatif
    print(f"\n{'─'*50}")
    print(f"{'Métrique':<25} {'A*':>10} {'Dijkstra':>10}")
    print(f"{'─'*50}")
    print(f"{'Distance (km)':<25} {dist_astar:>10} {dist_dijk:>10}")
    print(f"{'Noeuds explorés':<25} {exp_astar:>10} {exp_dijk:>10}")
    print(f"{'Temps (ms)':<25} {temps_astar:>10.4f} {temps_dijk:>10.4f}")
    print(f"{'─'*50}")

    print("\nConclusion :")
    if exp_astar < exp_dijk:
        print(f"  A* explore {exp_dijk - exp_astar} nœud(s) de moins grâce à l'heuristique.")

    elif exp_dijk < exp_astar:
        print(f"  Dijkstra explore {exp_astar - exp_dijk} nœud(s) de moins sur cette grille.")
    else:
        print("  Les deux algorithmes explorent le même nombre de nœuds sur cette grille.")

    visualiser_graphe(chemin_astar)

# Constantes
grille = construction_graphe()
depart = "Quebec"
arrivee = "Buffalo"


comparer()

   COMPARAISON A* vs DIJKSTRA

──────────────────────────────────────────────────
Métrique                          A*   Dijkstra
──────────────────────────────────────────────────
Distance (km)                   1050       1050
Noeuds explorés                    8          8
Temps (ms)                    0.0593     0.0144
──────────────────────────────────────────────────

Conclusion :
  Les deux algorithmes explorent le même nombre de nœuds sur cette grille.
Graphe généré : resultat_reseau_routier.pdf



### Analyse

Les différences de performance entre les algorithmes
Dans quels cas chacun est préférable